In [ ]:
!pip install titans-pytorch transformers -q


# Titans — Στόχος 2: Reasoning (Synthetic S-NIAH)
### Paper: *Titans: Learning to Memorize at Test Time* (Behrouz, Zhong, Mirrokni — Google Research, 2025)

## Task: Synthetic Multi-Key Needle-in-a-Haystack (S-NIAH)

Αυτό το notebook αναπαράγει το **Table 2 / Section 5.3** του paper —  
το S-NIAH benchmark που χρησιμοποιούν οι συγγραφείς για να αποδείξουν  
την υπεροχή του MAC Titan έναντι των baseline μοντέλων.

### Γιατί Synthetic (αντί για WikiText/BABILong)

Το MAC του paper εκπαιδεύτηκε σε **15-30 δισ. tokens** (FineWeb-Edu).  
Με free Colab (~2M tokens, ~5K iterations) ο MAC δεν μπορεί να αναπτύξει  
τον cross-segment memory mechanism που απαιτείται για το WikiText/BABILong task.  
Το synthetic task:
- Έχει **ελεγχόμενο θόρυβο** → μοντέλο μαθαίνει γρήγορα  
- Είναι **ακριβώς το S-NIAH benchmark** του paper (Table 2)  
- Αναδεικνύει ξεκάθαρα την υπεροχή του MAC σε μεγάλα context windows  

### Πειράματα

| Πείραμα | Μεταβλητή | Αντιστοιχεί σε |
|---|---|---|
| **Depth Test** | Θέση needle (10%/30%/50%/70%/90%) | "Lost in the Middle" |
| **Scaling Test** | Context length (128→2048 tokens) | Table 2 του paper |
| **Multi-Needle** | Αριθμός distractors (1/2/4/8) | S-NIAH-W/N variants |

### Αναμενόμενα αποτελέσματα (βάσει paper)
- **Baseline**: πτώση accuracy σε depth=0.5 (Lost in the Middle) και σε μεγάλα contexts  
- **MAC Titan**: σταθερή accuracy ανεξάρτητα από depth και context length  

### Χρήση
`MODE = "all"` → εκπαίδευση + αξιολόγηση (~30-50 λεπτά σε T4)  
`MODE = "train"` / `"eval"` → μόνο το αντίστοιχο μέρος


In [ ]:
# =============================================================================
# Titans: Learning to Memorize at Test Time — ΣΤΟΧΟΣ 2: Reasoning
# Synthetic Associative Recall Experiment
# Paper: Behrouz, Zhong, Mirrokni (Google Research, 2025)
#
# ΤΑΣΚ: Multi-Key Needle-in-a-Haystack (S-NIAH) — Table 2 του paper.
# Δημιουργούμε synthetic sequences με:
#   - Noise tokens (τυχαία, αδιάφορα)
#   - N key-value pairs παρεμβεβλημένα σε τυχαίες θέσεις (needles)
#   - Query στο τέλος: "ποιο είναι το value του key X;"
#   - Target: το αντίστοιχο value
#
# ΓΙΑΤΙ SYNTHETIC (αντί για WikiText/BABILong):
#   Το MAC του paper εκπαιδεύτηκε σε 15-30Β tokens (FineWeb-Edu) για να
#   αναπτύξει τον cross-segment memory mechanism. Με το WikiText (~2M tokens)
#   και free Colab, ο MAC δεν μαθαίνει αρκετά. Το synthetic task:
#   (α) Έχει ελεγχόμενο θόρυβο → το μοντέλο μαθαίνει γρήγορα
#   (β) Είναι ακριβώς το S-NIAH benchmark του paper (Table 2, Section 5.3)
#   (γ) Αναδεικνύει ξεκάθαρα την υπεροχή του MAC σε μεγάλα context lengths
#
# ΠΕΙΡΑΜΑΤΑ:
#   (Α) Depth Test: accuracy ανά θέση needle (10%/30%/50%/70%/90%) για
#       σταθερό context length — δείχνει το "Lost in the Middle" φαινόμενο
#   (Β) Scaling Test: accuracy ανά context length (256/512/1024/2048) —
#       δείχνει αντοχή MAC vs Baseline σε μεγαλύτερα windows
#   (Γ) Multi-needle: accuracy με 1/2/4/8 needles — δυσκολία
#
# ΑΠΟΤΕΛΕΣΜΑ ΠΟΥ ΑΝΑΜΕΝΕΤΑΙ (βάσει paper Table 2):
#   Baseline: πέφτει σε depth=0.5 (Lost in the Middle) και σε μεγάλα ctx
#   MAC: διατηρεί σχεδόν σταθερή accuracy ανεξάρτητα από depth και ctx length
#
# ΤΕΧΝΙΚΕΣ ΑΠΟΦΑΣΕΙΣ:
#   - seq[:,:-1] για ΑΜΦΟΤΕΡΑ → no data leakage
#   - Restricted CE: loss πάνω στα N_VALS logits, όχι 50K → γρήγορη σύγκλιση
#   - Curriculum MAC: 16→32→64→128→256 tokens
#   - Ίδιο parameter budget (parameter matching)
# =============================================================================

import os, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import get_linear_schedule_with_warmup
from titans_pytorch import MemoryAsContextTransformer

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
SEGMENT_LEN  = 16       # MAC segment size — μικρό ώστε να έχουμε πολλά segments
N_KEYS       = 32       # αριθμός διαφορετικών key-value pairs
TRAIN_CTX    = 128      # context length εκπαίδευσης (8 segments)

# Token ranges — όλα εντός VOCAB
VOCAB        = 150
NOISE_RANGE  = (0,   50)   # tokens 0-49: noise
KEY_RANGE    = (50,  50+N_KEYS)   # tokens 50-81: keys
VAL_RANGE    = (82,  82+N_KEYS)   # tokens 82-113: values
QUERY_MARKER = 114     # ειδικό token που σηματοδοτεί "ερώτηση"

NOISE_TOKENS = list(range(*NOISE_RANGE))
KEYS         = list(range(*KEY_RANGE))
VALS         = list(range(*VAL_RANGE))

# Eval grid
EVAL_DEPTHS      = [0.1, 0.3, 0.5, 0.7, 0.9]
EVAL_CTX_LENGTHS = [128, 256, 512, 1024, 2048]
EVAL_N_NEEDLES   = [1, 2, 4, 8]

CHECKPOINT_DIR = "checkpoints_synthetic"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

torch.manual_seed(42); random.seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"  ✅ GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("  ⚠️  CPU mode — για GPU: Runtime → T4 GPU → Save")

print(f"  Vocab={VOCAB}, N_KEYS={N_KEYS}, Segment={SEGMENT_LEN}, TrainCtx={TRAIN_CTX}")
print(f"  ln({N_KEYS})={math.log(N_KEYS):.3f} (τυχαίο επίπεδο)")

# ─────────────────────────────────────────────────────────────────────────────
# ΔΕΔΟΜΕΝΑ
# ─────────────────────────────────────────────────────────────────────────────

VAL_IDS_T = torch.tensor(VALS, device=device)   # (N_KEYS,) — για restricted CE

def make_batch(batch_size, ctx_len, n_needles=1, target_depth=None):
    """
    Δημιουργεί synthetic S-NIAH batch.

    Κάθε sequence:
      [noise...] [key_i val_i] [noise...] ... [noise...] [QMARK key_j] [val_j]
                  ↑needle i                                ↑query         ↑target (εκτός input)

    Args:
        batch_size:   μέγεθος batch
        ctx_len:      μήκος context (πολλαπλάσιο SEGMENT_LEN)
        n_needles:    αριθμός key-value pairs στο κείμενο (δυσκολία)
        target_depth: θέση του target needle (0.0-1.0). None = τυχαία

    Returns:
        seqs    (B, ctx_len+1) — τελευταίο token = απάντηση (εκτός input)
        targets (B,) ∈ {0..N_KEYS-1} — index του σωστού value
    """
    assert ctx_len % SEGMENT_LEN == 0
    seqs    = torch.randint(0, len(NOISE_TOKENS), (batch_size, ctx_len + 1),
                            device=device)   # αρχικά όλο noise
    targets = []

    q_len = 3   # QUERY_MARKER + key_token + answer_token

    for b in range(batch_size):
        # Επιλογή n_needles διαφορετικών key-value pairs
        idxs = random.sample(range(N_KEYS), min(n_needles, N_KEYS))
        pairs = [(KEYS[i], VALS[i]) for i in idxs]

        # Τοποθέτηση needles σε τυχαίες θέσεις (ή ελεγχόμενη depth για το target)
        max_pos = ctx_len - q_len - 2 * n_needles - 4
        available = list(range(0, max_pos, 3))
        positions = sorted(random.sample(available, min(n_needles, len(available))))

        # Το target needle τοποθετείται σε ελεγχόμενη θέση (depth test)
        if target_depth is not None and len(positions) > 0:
            target_pos = max(0, min(int(ctx_len * target_depth), max_pos))
            positions[0] = target_pos  # το 1ο needle είναι το target

        for pos, (k, v) in zip(positions, pairs):
            seqs[b, pos]   = k
            seqs[b, pos+1] = v

        # Query + answer στο τέλος
        # seq[-3] = QUERY_MARKER, seq[-2] = queried key, seq[-1] = answer (target)
        target_k, target_v = pairs[0]
        seqs[b, -3] = QUERY_MARKER
        seqs[b, -2] = target_k
        seqs[b, -1] = target_v   # αυτό αφαιρείται από input (seq[:,:-1])

        targets.append(VALS.index(target_v))  # index 0..N_KEYS-1

    return seqs, torch.tensor(targets, device=device)


def restricted_loss(logits, tgt_idx):
    """CE πάνω στα N_KEYS value logits — αντί για VOCAB."""
    restr = logits[:, -1, :][:, VAL_IDS_T]    # (B, N_KEYS)
    loss  = nn.CrossEntropyLoss()(restr, tgt_idx)
    acc   = (restr.argmax(-1) == tgt_idx).float().mean()
    return loss, acc


# ─────────────────────────────────────────────────────────────────────────────
# ΜΟΝΤΕΛΑ
# ─────────────────────────────────────────────────────────────────────────────

class BaselineTransformer(nn.Module):
    """
    Standard Transformer με full self-attention.
    Ο "Baseline" της σύγκρισης — βλέπει ολόκληρο το context ταυτόχρονα
    αλλά έχει positional bias (attention sink στις αρχικές θέσεις).
    """
    def __init__(self, dim=128, depth=4):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB, dim)
        self.pos_emb   = nn.Embedding(4096, dim)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=4, dim_feedforward=dim*4,
            batch_first=True, dropout=0.0)
        self.transformer = nn.TransformerEncoder(layer, num_layers=depth)
        self.to_logits   = nn.Linear(dim, VOCAB, bias=False)

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        h    = self.embedding(x) + self.pos_emb(pos)
        return self.to_logits(self.transformer(h))   # (B, T, VOCAB)


def build_mac_model(baseline_params):
    """MAC Titan — ψάχνει μεγαλύτερο dim εντός parameter budget."""
    for dim in range(256, 32, -8):
        m = MemoryAsContextTransformer(
            num_tokens             = VOCAB,
            dim                    = dim,
            depth                  = 4,
            segment_len            = SEGMENT_LEN,
            num_persist_mem_tokens = 4,
            num_longterm_mem_tokens= 8,
        ).to(device)
        if sum(p.numel() for p in m.parameters()) <= baseline_params:
            return m, dim
    raise RuntimeError("Δεν βρέθηκε MAC εντός budget.")


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


# ─────────────────────────────────────────────────────────────────────────────
# ΕΚΠΑΙΔΕΥΣΗ
# ─────────────────────────────────────────────────────────────────────────────

def _loop(model, batch_fn, label, total_iters, lr, ckpt, accum=4,
          min_iters=0, stop_loss=0.05, stop_acc=0.95):
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    n_upd = max(1, total_iters // accum)
    sch   = get_linear_schedule_with_warmup(opt, max(4, n_upd//8), n_upd)
    model.train(); opt.zero_grad()
    best_loss, best_state = float("inf"), None
    log_every = max(1, total_iters // 20)

    print(f"\n  [{label}] iters={total_iters} lr={lr}")
    for i in range(total_iters):
        seq, tgt = batch_fn()
        logits   = model(seq[:, :-1])
        loss, acc = restricted_loss(logits, tgt)
        (loss / accum).backward()
        if (i+1) % accum == 0:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
        if device.type == "cuda": torch.cuda.empty_cache()

        if i % log_every == 0 or i == total_iters - 1:
            print(f"  Iter {i:04d}/{total_iters} | loss={loss.item():.4f} | "
                  f"acc={acc.item()*100:.1f}% | lr={sch.get_last_lr()[0]:.2e}")
            if loss.item() < best_loss:
                best_loss  = loss.item()
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            if loss.item() < stop_loss and acc.item() > stop_acc and i >= min_iters:
                print(f"  ✅ Converged at iter {i}."); break

    if best_state: model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, ckpt))
    print(f"  Saved (best={best_loss:.4f}) → {CHECKPOINT_DIR}/{ckpt}")


def train_baseline(model):
    """
    Curriculum: 32→64→128 tokens.
    Ο Baseline μαθαίνει γρήγορα με full attention στα μικρά contexts.
    """
    print("\n" + "#"*60 + "\n  BASELINE — Curriculum Training\n" + "#"*60)
    stages = [(32,32,600),(64,24,600),(128,16,1500)]
    opt    = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    total  = sum(s[2] for s in stages)
    sch    = get_linear_schedule_with_warmup(opt, max(4, total//32), total//4)
    model.train(); opt.zero_grad()
    best_loss, best_state, g = float("inf"), None, 0

    for ctx, bs, max_it in stages:
        print(f"\n  Stage ctx={ctx} bs={bs} iters={max_it}")
        for li in range(max_it):
            n = random.choice([1,2])     # 1-2 needles κατά εκπαίδευση
            seq, tgt = make_batch(bs, ctx, n)
            logits   = model(seq[:, :-1])
            loss, acc = restricted_loss(logits, tgt)
            (loss/4).backward()
            if (g+1)%4==0:
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step(); sch.step(); opt.zero_grad()
            if g%200==0:
                print(f"  Iter {g:04d} | loss={loss.item():.4f} | "
                      f"acc={acc.item()*100:.1f}% | ctx={ctx}")
                if loss.item()<best_loss:
                    best_loss=loss.item()
                    best_state={k:v.clone() for k,v in model.state_dict().items()}
                if loss.item()<0.04 and acc.item()>0.98 and ctx==TRAIN_CTX:
                    print("  ✅ Converged early."); break
            g+=1
        if device.type=="cuda": torch.cuda.empty_cache()

    if best_state: model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR,"baseline.pt"))
    print(f"  Saved → {CHECKPOINT_DIR}/baseline.pt")


def train_mac(model):
    """
    Curriculum για MAC: 16→32→64→128 tokens.
    Το MAC χρειάζεται curriculum γιατί το neural memory mechanism μαθαίνει
    cross-segment retrieval σταδιακά — ξεκινά από 1 segment και ανεβαίνει.
    """
    print("\n" + "#"*60 + "\n  MAC TITAN — Curriculum Training\n" + "#"*60)
    stages = [(16,32,800),(32,24,800),(64,16,1000),(128,8,3000)]
    opt    = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    total  = sum(s[2] for s in stages)
    sch    = get_linear_schedule_with_warmup(opt, max(4, total//32), total//4)
    model.train(); opt.zero_grad()
    best_loss, best_state, g = float("inf"), None, 0

    for ctx, bs, max_it in stages:
        print(f"\n  Stage ctx={ctx} bs={bs} iters={max_it}")
        for li in range(max_it):
            n = 1 if ctx<=32 else random.choice([1,2])
            seq, tgt = make_batch(bs, ctx, n)
            logits   = model(seq[:, :-1])
            loss, acc = restricted_loss(logits, tgt)
            (loss/4).backward()
            if (g+1)%4==0:
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step(); sch.step(); opt.zero_grad()
            if g%200==0:
                print(f"  Iter {g:04d} | loss={loss.item():.4f} | "
                      f"acc={acc.item()*100:.1f}% | ctx={ctx}")
                if loss.item()<best_loss:
                    best_loss=loss.item()
                    best_state={k:v.clone() for k,v in model.state_dict().items()}
                if loss.item()<0.04 and acc.item()>0.98 and ctx==TRAIN_CTX:
                    print("  ✅ Converged early."); break
            g+=1
        if device.type=="cuda": torch.cuda.empty_cache()

    if best_state: model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR,"mac.pt"))
    print(f"  Saved → {CHECKPOINT_DIR}/mac.pt")


# ─────────────────────────────────────────────────────────────────────────────
# ΑΞΙΟΛΟΓΗΣΗ
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def eval_grid(model, label, ctx_lengths, depths=None, n_needles=1,
              trials=10, bs=8):
    """Αξιολόγηση ανά (ctx_len, depth). Επιστρέφει matrix [depths x ctx_lengths]."""
    model.eval()
    depths = depths or EVAL_DEPTHS
    R = np.zeros((len(depths), len(ctx_lengths)))
    print(f"\n  [{label}] Evaluation (n_needles={n_needles})")
    for ci, ctx in enumerate(ctx_lengths):
        if ctx % SEGMENT_LEN != 0:
            print(f"  ctx={ctx} δεν είναι πολλαπλάσιο SEGMENT_LEN={SEGMENT_LEN}, skip")
            continue
        for di, d in enumerate(depths):
            correct = total = 0
            for _ in range(trials):
                seq, tgt = make_batch(bs, ctx, n_needles, target_depth=d)
                r = model(seq[:, :-1])[:, -1, :][:, VAL_IDS_T]
                correct += (r.argmax(-1) == tgt).sum().item()
                total   += bs
                if device.type == "cuda": torch.cuda.empty_cache()
            acc = correct / total; R[di, ci] = acc
            m = "✅" if acc>=0.7 else ("~" if acc>=0.4 else "❌")
            print(f"  {m} ctx={ctx:5d} | depth={d:.1f} → {acc*100:.1f}%")
    return R


@torch.no_grad()
def eval_needles(model, label, ctx, n_needles_list, trials=10, bs=8):
    """Αξιολόγηση ανά αριθμό needles — δυσκολία."""
    model.eval()
    results = []
    for n in n_needles_list:
        correct = total = 0
        for _ in range(trials):
            seq, tgt = make_batch(bs, ctx, n)
            r = model(seq[:, :-1])[:, -1, :][:, VAL_IDS_T]
            correct += (r.argmax(-1) == tgt).sum().item()
            total   += bs
        acc = correct / total; results.append(acc)
        m = "✅" if acc>=0.7 else ("~" if acc>=0.4 else "❌")
        print(f"  {m} [{label}] ctx={ctx} n_needles={n} → {acc*100:.1f}%")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────────────────────

def _hm(ax, data, title, xlabels, ylabels, xl, yl):
    im = ax.imshow(data, cmap="RdYlGn", vmin=0, vmax=1,
                   origin="lower", aspect="auto", interpolation="nearest")
    ax.set(xticks=range(len(xlabels)), yticks=range(len(ylabels)))
    ax.set_xticklabels(xlabels); ax.set_yticklabels([f"{v:.1f}" for v in ylabels])
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title, fontsize=10)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v=data[i,j]
            ax.text(j,i,f"{v*100:.0f}%",ha="center",va="center",fontsize=9,
                    color="black" if 0.25<v<0.75 else "white",fontweight="bold")
    return im


def save_plots(base_R, mac_R, ctx_lengths, depths):
    xl = [f"{c}" for c in ctx_lengths]

    # ── Heatmaps ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("S-NIAH — Accuracy ανά needle depth × context length\n"
                 "(Baseline vs MAC Titan)", fontweight="bold")
    for ax, mat, ttl in zip(axes, [base_R, mac_R],
                            ["Baseline Transformer", "MAC Titan"]):
        im = _hm(ax, mat, ttl, xl, depths, "Context length (tokens)", "Needle depth (%)")
    fig.colorbar(im, ax=axes.tolist(), label="Accuracy")
    fig.savefig("depth_heatmap.png", dpi=150, bbox_inches="tight"); plt.close(fig)
    print("  Saved → depth_heatmap.png")

    # ── Depth curves (Lost-in-the-Middle) ────────────────────────────────────
    n = min(3, len(ctx_lengths))
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=True)
    if n == 1: axes = [axes]
    fig.suptitle("Needle-in-a-Haystack: Accuracy vs Needle Depth\n"
                 "'Lost in the Middle' — Baseline vs MAC Titan",
                 fontweight="bold")
    for j, ctx in enumerate(ctx_lengths[:n]):
        ax = axes[j]
        ax.plot(depths, base_R[:, j], "o--", color="#e74c3c",
                lw=2, ms=7, label="Baseline Transformer")
        ax.plot(depths, mac_R[:, j], "o-",  color="#2ecc71",
                lw=2, ms=7, label="MAC Titan")
        ax.axhline(1/N_KEYS, color="#888", lw=1.2, ls=":",
                   label=f"Τυχαίο (1/{N_KEYS})")
        ax.set_title(f"context = {ctx} tokens\n({ctx//SEGMENT_LEN} segments)")
        ax.set_xlabel("Needle depth (% context)")
        ax.set_ylim(-0.05, 1.05); ax.grid(alpha=0.3)
        ax.set_xticks(depths); ax.set_xticklabels([f"{d:.1f}" for d in depths])
        if j == 0: ax.set_ylabel("Accuracy"); ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("depth_curves.png", dpi=150, bbox_inches="tight"); plt.close(fig)
    print("  Saved → depth_curves.png")

    # ── Scaling: mean accuracy vs context length ─────────────────────────────
    bm = base_R.mean(0)*100; mm = mac_R.mean(0)*100
    x  = np.arange(len(ctx_lengths)); w = 0.32
    fig, ax = plt.subplots(figsize=(8, 4))
    b1=ax.bar(x-w/2, bm, w, label="Baseline Transformer", color="#e74c3c", alpha=0.9)
    b2=ax.bar(x+w/2, mm, w, label="MAC Titan",           color="#2ecc71", alpha=0.9)
    ax.axhline(100/N_KEYS, color="#888", lw=1.5, ls=":",
               label=f"Τυχαίο ({100//N_KEYS}%)")
    for bar in list(b1)+list(b2):
        h=bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+1, f"{h:.0f}%",
                ha="center", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels([f"{c}" for c in ctx_lengths])
    ax.set_xlabel("Context length (tokens)"); ax.set_ylabel("Mean Accuracy (%)")
    ax.set_ylim(0, 115)
    ax.set_title("Scaling Test: Mean Accuracy ανά Context Length\n"
                 "(avg over all depths)")
    ax.legend(); fig.tight_layout()
    fig.savefig("scaling_bar.png", dpi=150, bbox_inches="tight"); plt.close(fig)
    print("  Saved → scaling_bar.png")


def save_needle_plot(base_acc, mac_acc, ctx):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(EVAL_N_NEEDLES, [a*100 for a in base_acc], "o--",
            color="#e74c3c", lw=2, ms=8, label="Baseline Transformer")
    ax.plot(EVAL_N_NEEDLES, [a*100 for a in mac_acc], "o-",
            color="#2ecc71", lw=2, ms=8, label="MAC Titan")
    ax.axhline(100/N_KEYS, color="#888", lw=1.2, ls=":",
               label=f"Τυχαίο ({100//N_KEYS}%)")
    ax.set_xlabel("Αριθμός Needles (distractors)")
    ax.set_ylabel("Accuracy (%)"); ax.set_ylim(-5, 105)
    ax.set_xticks(EVAL_N_NEEDLES)
    ax.set_title(f"Multi-Needle Test (ctx={ctx} tokens)\n"
                 "Αντοχή σε αυξανόμενο αριθμό distractors")
    ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
    fig.savefig("needle_curve.png", dpi=150, bbox_inches="tight"); plt.close(fig)
    print("  Saved → needle_curve.png")


def print_table(base_R, mac_R, ctx_lengths, depths):
    xl = "  ".join(f"{c:>5}" for c in ctx_lengths)
    print(f"\n{'='*70}")
    print(f"  S-NIAH Results — Accuracy ανά depth × context length")
    print(f"{'='*70}")
    print(f"  {'depth':>6} | {'Baseline':^{len(xl)}} | {'MAC Titan':^{len(xl)}}")
    print(f"  {'':6} | {xl} | {xl}")
    print(f"  {'-'*70}")
    for di, d in enumerate(depths):
        br = "  ".join(f"{base_R[di,ci]*100:>5.1f}%" for ci in range(len(ctx_lengths)))
        mr = "  ".join(f"{mac_R[di,ci]*100:>5.1f}%" for ci in range(len(ctx_lengths)))
        print(f"  {d:.1f}    | {br} | {mr}")
    print(f"\n  Mean:")
    br = "  ".join(f"{base_R[:,ci].mean()*100:>5.1f}%" for ci in range(len(ctx_lengths)))
    mr = "  ".join(f"{mac_R[:,ci].mean()*100:>5.1f}%" for ci in range(len(ctx_lengths)))
    print(f"  {'':6} | {br} | {mr}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main(MODE="all"):
    print(f"\n{'='*60}")
    print(f"  Titans Synthetic S-NIAH | MODE={MODE} | device={device}")
    print(f"{'='*60}\n")

    baseline = BaselineTransformer(dim=128, depth=4).to(device)
    bp       = count_params(baseline)
    mac, mac_dim = build_mac_model(bp)

    print(f"  Baseline params: {bp:,}")
    print(f"  MAC Titan params: {count_params(mac):,}  (dim={mac_dim})\n")

    bpath = os.path.join(CHECKPOINT_DIR, "baseline.pt")
    mpath = os.path.join(CHECKPOINT_DIR, "mac.pt")

    if MODE in ("train", "all"):
        train_baseline(baseline)
        train_mac(mac)
    else:
        for name, model, path in [("Baseline",baseline,bpath),("MAC",mac,mpath)]:
            if not os.path.exists(path):
                raise FileNotFoundError(f"{path} δεν βρέθηκε. Τρέξε πρώτα train.")
            model.load_state_dict(torch.load(path, map_location=device))
            print(f"  ✅ {name} ← {path}")

    if MODE in ("eval", "all"):
        # ── Eval context lengths (μόνο πολλαπλάσια SEGMENT_LEN) ────────────
        valid_ctx = [c for c in EVAL_CTX_LENGTHS if c % SEGMENT_LEN == 0]

        print("\n  === DEPTH TEST (1 needle) ===")
        base_R = eval_grid(baseline, "Baseline", valid_ctx,
                           EVAL_DEPTHS, n_needles=1, trials=8, bs=8)
        mac_R  = eval_grid(mac,      "MAC Titan", valid_ctx,
                           EVAL_DEPTHS, n_needles=1, trials=8, bs=8)

        print("\n  === MULTI-NEEDLE TEST ===")
        base_n = eval_needle_multi(baseline, "Baseline", TRAIN_CTX, trials=8, bs=8)
        mac_n  = eval_needle_multi(mac,      "MAC Titan", TRAIN_CTX, trials=8, bs=8)

        # Save .npy
        for nm, arr in [("base_depth",base_R),("mac_depth",mac_R)]:
            np.save(os.path.join(CHECKPOINT_DIR,f"{nm}.npy"), arr)

        print_table(base_R, mac_R, valid_ctx, EVAL_DEPTHS)
        save_plots(base_R, mac_R, valid_ctx, EVAL_DEPTHS)
        save_needle_plot(base_n, mac_n, TRAIN_CTX)

        print(f"\n  ✅ Ολοκλήρωση.")
        print(f"  Plots: depth_heatmap.png, depth_curves.png, "
              f"scaling_bar.png, needle_curve.png")


@torch.no_grad()
def eval_needle_multi(model, label, ctx, trials=8, bs=8):
    model.eval()
    results = []
    print(f"\n  [{label}] Multi-needle (ctx={ctx})")
    for n in EVAL_N_NEEDLES:
        if n > N_KEYS: break
        correct = total = 0
        for _ in range(trials):
            seq, tgt = make_batch(bs, ctx, n)
            r = model(seq[:,:-1])[:,-1,:][:,VAL_IDS_T]
            correct += (r.argmax(-1)==tgt).sum().item(); total+=bs
        acc = correct/total; results.append(acc)
        m = "✅" if acc>=0.7 else ("~" if acc>=0.4 else "❌")
        print(f"  {m} n_needles={n:2d} → {acc*100:.1f}%")
    return results


# Εκτέλεση: βλ. επόμενο cell


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ΕΚΤΕΛΕΣΗ
#   "all"   → εκπαίδευση + αξιολόγηση  (~30-50 λεπτά σε T4 GPU)
#   "train" → μόνο εκπαίδευση
#   "eval"  → μόνο αξιολόγηση (χρειάζεται checkpoints)
# ─────────────────────────────────────────────────────────────────────────────
MODE = "all"
main(MODE)

## Αποτελέσματα — Plots

In [ ]:
from IPython.display import Image, display
import os

for fname, title in [
    ("depth_heatmap.png",  "Depth Test — Heatmap (Baseline vs MAC Titan)"),
    ("depth_curves.png",   "Depth Curves — Lost in the Middle"),
    ("scaling_bar.png",    "Scaling Test — Mean Accuracy ανά Context Length"),
    ("needle_curve.png",   "Multi-Needle Test — Αντοχή σε Distractors"),
]:
    if os.path.exists(fname):
        print(f"\n### {title}")
        display(Image(fname))
    else:
        print(f"  (Δεν βρέθηκε: {fname})")

## (Προαιρετικό) Αποθήκευση στο Google Drive

In [ ]:
from google.colab import drive
import shutil, datetime

drive.mount("/content/drive")
ts   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
dest = f"/content/drive/MyDrive/Titans_Synthetic_NIAH_{ts}"
os.makedirs(dest, exist_ok=True)

for f in ["depth_heatmap.png","depth_curves.png",
          "scaling_bar.png","needle_curve.png"]:
    if os.path.exists(f):
        shutil.copy2(f, dest)

if os.path.isdir(CHECKPOINT_DIR):
    shutil.copytree(CHECKPOINT_DIR,
                    os.path.join(dest, CHECKPOINT_DIR),
                    dirs_exist_ok=True)
print(f"Αποθηκεύτηκαν στο: {dest}")